# SLM Fine-tuning V2 — RoBERTa Classifier Head Only

## Triết lý thiết kế

Notebook này triển khai một chiến lược huấn luyện **2 bước** theo đúng tinh thần MRCD:

### Bước 1: Tải kiến trúc mô hình lõi (Tích hợp RoBERTa)
- Sử dụng `RoBERTaForSequenceClassification` từ HuggingFace đã được **pre-train**.
- Đỉnh mô hình được thiết lập thành lớp phân loại nhị phân (**Binary Classification**) cho 2 nhãn: **Real / Fake**.
- Không cần xây dựng mô hình từ con số 0.

### Bước 2: Huấn luyện "Mồi" (Khởi tạo ban đầu)
- Huấn luyện trên tập **"Sự kiện quá khứ" (Past Events)** — dữ liệu tin tức cũ đã được gán nhãn.
- **AdamW** optimizer, `batch_size=32`, `weight_decay=1e-4`, `lr=1e-3`.

> ⚠️ **CẢNH BÁO QUAN TRỌNG:**
> Learning Rate `1e-3` là **RẤT CAO** đối với các mô hình Transformer được pre-train. Với LR này, **BẮT BUỘC phải đóng băng (freeze) toàn bộ phần thân mạng RoBERTa** và chỉ mở khoá huấn luyện cho **Classifier Head** ở trên cùng.
>
> Nếu lỡ mở khoá toàn bộ trọng số của RoBERTa mà giữ nguyên mức LR `1e-3`, mô hình sẽ bị **phá hủy kiến thức** (catastrophic forgetting) và học sai hoàn toàn.
>
> Mức LR `1e-3` chỉ an toàn cho classifier head (khởi tạo ngẫu nhiên), KHÔNG an toàn cho backbone đã pre-train.

---

**Steps:**
1. Environment Setup
2. Data Path Configuration
3. Load Data from CSV files
4. Dataset Preparation
5. Load Pre-trained RoBERTa + Freeze Backbone
6. Seed Training (Classifier Head Only)
7. Model Evaluation on Test Set
8. Save Trained Model

## 1. Environment Setup

In [ ]:
# %pip install torch transformers accelerate sentencepiece pandas scikit-learn tqdm seaborn matplotlib

In [ ]:
# --- 1. Environment Setup ---
import os
import sys
import json
import re
import random
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple

# Suppress warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

# Transformers
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# Metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

## 2. Data Path Configuration

In [ ]:
# --- Data Paths Configuration ---
DATA_DIR = r"/kaggle/input/datasets/chinhde/twitter15-16"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
MODEL_SAVE_PATH = "/kaggle/working/RoBERTaV2"

print(f"Data Directory: {DATA_DIR}")
print(f"Train CSV: {TRAIN_CSV}")
print(f"Test CSV: {TEST_CSV}")
print(f"Model Save Path: {MODEL_SAVE_PATH}")

for fpath, name in [(TRAIN_CSV, 'Train'), (TEST_CSV, 'Test')]:
    if os.path.exists(fpath):
        print(f"{name} file exists: {fpath}")
    else:
        print(f"{name} file not found: {fpath}")

## 3. Data Loading from CSV

In [ ]:
# --- Shared Preprocessing and Data Loading ---
def preprocess_text(text: str) -> str:
    """
    Shared preprocessing used by both SLM and MRCD.
    - Lowercase
    - Remove URLs
    - Remove @mentions
    - Keep alphanumeric, underscore, # and spaces
    - Normalize spaces
    """
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^\w\s#]", " ", text)
    text = " ".join(text.split())
    return text


def load_data_from_csv(train_csv, test_csv):
   
    def load_csv_file(filepath):
        if not os.path.exists(filepath):
            print(f"File not found: {filepath}")
            return [], []

        try:
            df = pd.read_csv(filepath)
            texts = [preprocess_text(t) for t in df['text'].astype(str).tolist()]
            labels = [0 if label.lower() in ['true', 'non-rumor'] else 1 for label in df['label'].astype(str).tolist()]
            print(f"Loaded {len(texts)} samples from {os.path.basename(filepath)}")
            print(f"  Label distribution: {pd.Series(labels).value_counts().to_dict()}")
            return texts, labels
        except Exception as e:
            print(f"Error loading {filepath}: {e}")
            return [], []

    print("Loading data from CSV files...")
    train_texts, train_labels = load_csv_file(train_csv)
    test_texts, test_labels = load_csv_file(test_csv)


    print(f"Train size: {len(train_texts)}")
    print(f"Test size: {len(test_texts)}")

    return train_texts, train_labels, test_texts, test_labels
# Load data
train_texts, train_labels, test_texts, test_labels = load_data_from_csv(
    TRAIN_CSV, TEST_CSV
)

In [ ]:
# --- Data Distribution Analysis ---
if len(train_texts) > 0 and len(test_texts) > 0:
    df_train = pd.DataFrame({'label': train_labels})
    df_test = pd.DataFrame({'label': test_labels})

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    sns.countplot(x='label', data=df_train, palette='viridis')
    plt.title(f"Training Set (train, n={len(train_texts)})\n0=Truth, 1=Rumor")
    plt.xlabel("Label")
    plt.ylabel("Count")

    plt.subplot(1, 2, 2)
    sns.countplot(x='label', data=df_test, palette='viridis')
    plt.title(f"Test Set (n={len(test_texts)})\n0=Truth, 1=Rumor")
    plt.xlabel("Label")
    plt.ylabel("Count")

    plt.tight_layout()
    plt.show()

    train_lengths = [len(t.split()) for t in train_texts]
    test_lengths = [len(t.split()) for t in test_texts]

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.hist(train_lengths, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
    plt.title(f"Training Text Length\n(mean={np.mean(train_lengths):.0f}, median={np.median(train_lengths):.0f})")
    plt.xlabel("Words per text")
    plt.ylabel("Frequency")

    plt.subplot(1, 2, 2)
    plt.hist(test_lengths, bins=30, color='coral', alpha=0.7, edgecolor='black')
    plt.title(f"Test Text Length\n(mean={np.mean(test_lengths):.0f}, median={np.median(test_lengths):.0f})")
    plt.xlabel("Words per text")
    plt.ylabel("Frequency")

    plt.tight_layout()
    plt.show()
else:
    print("Insufficient data for visualization")

## 4. Dataset Preparation

In [ ]:
# --- PyTorch Dataset ---
class FakeNewsDataset(Dataset):
    """PyTorch Dataset for fake news detection"""
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = preprocess_text(self.texts[idx])
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

print("Loading tokenizer...")
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

train_dataset = FakeNewsDataset(train_texts, train_labels, tokenizer)
test_dataset = FakeNewsDataset(test_texts, test_labels, tokenizer)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Datasets created:")
print(f"  Train: {len(train_dataset)} samples (Batch size: {BATCH_SIZE})")
print(f"  Test:  {len(test_dataset)} samples (Batch size: {BATCH_SIZE})")

## 5. Load Pre-trained RoBERTa & Freeze Backbone

### Chiến lược Freeze:
- **Toàn bộ thân mạng RoBERTa** (encoder backbone) sẽ bị **đóng băng** (`requires_grad = False`).
- Chỉ **Classifier Head** (2 lớp dense cuối) mới được huấn luyện.

> ⚠️ **VÌ SAO PHẢI FREEZE?**
>
> Learning rate `1e-3` là **mức rất cao** — nó phù hợp để huấn luyện các lớp khởi tạo ngẫu nhiên (như classifier head), nhưng nếu áp dụng cho backbone đã pre-train, nó sẽ **phá hủy kiến thức** (catastrophic forgetting).
>
> Backbone RoBERTa đã được pre-train hàng tuần trên hàng TB dữ liệu. Với LR `1e-3`, gradient quá lớn sẽ khiến trọng số di chuyển quá xa khỏi vùng tối ưu, dẫn đến mô hình mất hoàn toàn khả năng hiểu ngôn ngữ.
>
> Nếu muốn fine-tune cả backbone, cần dùng LR thấp hơn nhiều (ví dụ `1e-5` đến `2e-5`).

In [ ]:
# --- Load Model ---
print("Loading pre-trained RoBERTa-base model...")
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2,     # Binary: Real (0) vs Fake (1)
)

# =====================================================================
# ⚠️ FREEZE BACKBONE — CHỈ HUẤN LUYỆN CLASSIFIER HEAD
# =====================================================================
# Đóng băng toàn bộ thân mạng RoBERTa encoder
for param in model.roberta.parameters():
    param.requires_grad = False

# Classifier head vẫn requires_grad = True (mặc định)
# Kiểm tra bao nhiêu parameters đang trainable
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\n{'='*60}")
print(f"📊 Model Parameter Summary:")
print(f"{'='*60}")
print(f"  Total params:     {total_params:>12,}")
print(f"  Trainable params: {trainable_params:>12,}  ← Classifier Head Only")
print(f"  Frozen params:    {frozen_params:>12,}  ← RoBERTa Backbone")
print(f"  Trainable ratio:  {trainable_params/total_params*100:.2f}%")
print(f"{'='*60}")

# Liệt kê các layer đang trainable
print("\n🔓 Trainable layers:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.shape}")

model.to(device)
print(f"\nModel loaded on {device}")

## 6. Seed Training — Classifier Head Only

### Hyperparameters (theo yêu cầu MRCD):
| Parameter | Value | Ghi chú |
|---|---|---|
| Optimizer | **AdamW** | |
| Batch size | **32** | |
| Weight decay | **1e-4** | |
| Learning rate | **1e-3** | ⚠️ Rất cao — chỉ an toàn cho classifier head |
| Epochs | 4 | Có thể tăng vì chỉ train head |

> Vì chỉ train classifier head (rất ít parameters), training sẽ **rất nhanh** so với full fine-tuning.

In [ ]:
# --- Training Configuration ---
# ⚠️ LR = 1e-3 — CHỈ AN TOÀN KHI BACKBONE ĐÃ BỊ FREEZE!
# Nếu mở khoá backbone mà giữ LR này → catastrophic forgetting!
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 4
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0

print("Training configuration (Seed Training — Head Only):")
print(f"  Learning Rate:  {LEARNING_RATE}  ⚠️ High LR — safe only because backbone is FROZEN")
print(f"  Weight Decay:   {WEIGHT_DECAY}")
print(f"  Epochs:         {EPOCHS}")
print(f"  Batch Size:     {BATCH_SIZE}")
print(f"  Warmup Ratio:   {WARMUP_RATIO}")
print(f"  Max Seq Length: 128")

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

# Chỉ truyền parameters có requires_grad=True cho optimizer
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"\nOptimizer & Scheduler:")
print(f"  Total steps:  {total_steps}")
print(f"  Warmup steps: {warmup_steps}")

In [ ]:
# --- Class Weights cho xử lý mất cân bằng ---
n_truth = sum(1 for l in train_labels if l == 0)
n_rumor = sum(1 for l in train_labels if l == 1)
n_total = n_truth + n_rumor

class_weights = torch.tensor([
    n_total / (2 * n_truth),   # Truth
    n_total / (2 * n_rumor),   # Rumor
], dtype=torch.float).to(device)

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

print(f"Class weights — Truth: {class_weights[0]:.4f}, Rumor: {class_weights[1]:.4f}")
print(f"Total steps: {total_steps} | Warmup: {warmup_steps}")

In [ ]:
# --- Training Loop ---
def train_slm(model, train_loader, optimizer, scheduler, device, epochs, loss_fn):
    history = {'loss': []}
    best_train_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        total_loss   = 0.0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

        for batch in progress_bar:
            optimizer.zero_grad()
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss    = loss_fn(outputs.logits, labels)   # weighted loss
            total_loss += loss.item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)
        history['loss'].append(avg_loss)

        if avg_loss < best_train_loss:
            best_train_loss = avg_loss
            torch.save(model.state_dict(), "best_model_v2.pt")
            print(f"  Train Loss: {avg_loss:.4f} ✅ Best saved")
        else:
            print(f"  Train Loss: {avg_loss:.4f} (best: {best_train_loss:.4f})")

    model.load_state_dict(torch.load("best_model_v2.pt"))
    print("\n✅ Training complete — best model restored.")
    return history


history = train_slm(
    model=model,
    train_loader=train_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    epochs=EPOCHS,
    loss_fn=loss_fn,
)

In [ ]:
# --- Plot Training History ---
plt.figure(figsize=(8, 5))
plt.plot(history['loss'], label='Training Loss', color='red', marker='o')
plt.title('Seed Training Loss per Epoch (Classifier Head Only, LR=1e-3)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 7. Model Evaluation on Test Set

In [ ]:
# --- Test Set Evaluation ---
def evaluate_on_test_set(model, test_loader, device):
    """Evaluate model on test set"""
    model.eval()
    preds_all = []
    labels_all = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    return labels_all, preds_all

test_labels_true, test_preds = evaluate_on_test_set(model, test_loader, device)

test_acc = accuracy_score(test_labels_true, test_preds)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels_true, test_preds, average='binary')

print("\nTest Set Evaluation (V2 — Head Only, LR=1e-3):")
print(f"  Accuracy:  {test_acc:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print("\nClassification Report:")
print(classification_report(test_labels_true, test_preds, target_names=['Truth', 'Rumor'], digits=4))

In [ ]:
# --- Confusion Matrix ---
cm = confusion_matrix(test_labels_true, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Truth', 'Rumor'],
            yticklabels=['Truth', 'Rumor'])
plt.title('Confusion Matrix — V2 (Head Only, LR=1e-3)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 8. Save Trained Model

In [ ]:
# --- Save Model & Tokenizer ---
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
model.save_pretrained(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)
print(f"\n✅ Model and tokenizer saved to: {MODEL_SAVE_PATH}")
print(f"\nFiles saved:")
for f in os.listdir(MODEL_SAVE_PATH):
    fpath = os.path.join(MODEL_SAVE_PATH, f)
    size_mb = os.path.getsize(fpath) / 1024 / 1024
    print(f"  {f}: {size_mb:.2f} MB")

## 9. So sánh V1 vs V2

| | V1 (Full Fine-tuning) | V2 (Head Only) |
|---|---|---|
| **Learning Rate** | `1e-5` | `1e-3` |
| **Weight Decay** | `0.01` | `1e-4` |
| **Trainable params** | ~125M (100%) | ~590K (~0.5%) |
| **Backbone** | Unfrozen | ❄️ Frozen |
| **Catastrophic Forgetting Risk** | Thấp (LR nhỏ) | Không có (backbone frozen) |
| **Training Speed** | Chậm | ⚡ Rất nhanh |
| **Mục đích** | Fine-tune toàn bộ | Huấn luyện "mồi" nhanh |

V2 là bước khởi tạo ban đầu cho SLM trong hệ thống MRCD. Sau bước này, SLM đã có nền tảng phân biệt thật/giả cơ bản, sẵn sàng cho các vòng lặp MRCD tiếp theo.